# BSC FGI Scheduler
Simulates AP rollout into FGI, labor completion, task-driven moves, delivery closure, trace export, and KPI/flow reporting.

## 1. Imports and configuration
Library imports and notebook-wide configuration.

In [1]:
## LIBRARY IMPORTS
import pandas as pd
import numpy as np
import datetime as dt
from datetime import timedelta
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import math
from math import dist
import copy
import os
from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill, Border, Side, Alignment
from openpyxl.utils import get_column_letter
from pathlib import Path
import shutil
import tempfile

## 2. File paths and input loading
Notebook-relative paths to inputs in `data/staged/` and outputs in `output/`.

In [2]:
# FILEPATHS FOR IMPORT
rootpath = Path.cwd()
rootpath = Path.cwd().parent

filepaths = {
    'FAstatus': rootpath / 'data' / 'raw' / 'FA_Status_FGI_Handoff.xlsx',
    'FGI_Locations': rootpath / 'data' / 'raw' / 'FGI_Locations_wPriority.xlsx',
    'Nodes': rootpath / 'data' / 'raw' / 'Nodes.xlsx',
    'Move Times': rootpath / 'data' / 'staged' / 'move_times' / 'move_time_estimation.xlsx',
    'Paint Schedule': rootpath / 'data' / 'raw' / 'paint_schedules.xlsx'
}


In [3]:
print(rootpath)

/Users/jishnughosh/Documents/GitHub/BSC-FGI_Scheduler


## 3. Constants and scheduling assumptions
FGI team list, BTG-by-team mapping, and the BTG-conversion helper used by every AP.

In [4]:
FGI_TEAMS = ['structure', 'systems', 'declam', 'test']
BTG_TYPES = ['tot', 'p0', 'p1', 'p2', 'p3', 'engines', 'doors', 'test']

BTG_TYPES_BY_LABOR = { # relationships ONLY, NOT 1-to-1 btg conversion
    'structure': ['tot'],
    'systems': ['p2'],
    'declam': ['p3', 'engines'],
    'test': ['test']
}
FGI_TEAMS_BY_BTG_TYPE = { # relationships ONLY, NOT 1-to-1 btg conversion
    'tot': ['structure', 'systems', 'declam', 'test'],
    'FGI_tot': ['structure'],
    'p2': ['systems'],
    'p3': ['declam'],
    'engines': ['declam'],
    'test': ['test']
}
# set non-associated BTG types to None in FGI_TEAMS_BY_BTG_TYPE
for type in BTG_TYPES: 
    if type not in FGI_TEAMS_BY_BTG_TYPE.keys():
        FGI_TEAMS_BY_BTG_TYPE[type] = None


In [5]:
## FGI BTG CONVERSION FUNCTION
def get_FGI_BTG(faro_btg):
    FGI_btg = {'FGI_tot': faro_btg['tot'] - faro_btg['p0'] - faro_btg['p1'], 
                        'structure': math.ceil(0.1 * (faro_btg['tot'] - faro_btg['p0'] - faro_btg['p1'])), 
                        'systems': math.ceil(faro_btg['p2']),
                        'declam': math.ceil(faro_btg['engines']),
                        'test': faro_btg['test']
                        }
    return FGI_btg

Live-state inputs from `data/staged/FGI_liveState.xlsx` define the AP, location, and labor dataframes consumed by the scheduler. Re-run the build cells below if the live state workbook changes.

## 4. Run conditions
Toggles and constants that control the run window, console verbosity, and the new-FA online flag.

In [6]:
## RUN CONDITIONS
# a) planning horizon
STARTDATE='2024-08-13'
ENDDATE='2026-3-10'
FORECAST_UNTIL_COMPLETION = True

# b) location parameters

NEW_FA_ONLINE = True
DISCONTINUED_LOCATIONS = ['S1', 'S2', 'S3', 'Spur']
INCLUDE_FA_LOCATIONS = True

# c) schedule parameters
PLANNED_BUFFER_DAYS = []
IMPORT_PAINT_SCHEDULE = False

# d) queuing methods
methods = ['FIFO', 'SPT', 'LPT', 'Critical Path', 'Shortest Queue']
SELECTED_METHOD = 'FIFO'

# e) output/export parameters
EXPORT_TO_EXCEL = True
EXPORT_PATH = rootpath / 'output'
EXPORT_TO_FGI_LIVESTATE = True


In [7]:
## UTILITY FUNCTIONS (for formatting output & debugging)
CODECELL_OUTPUT = True

def line(char='—', length = 100):
    if not CODECELL_OUTPUT:
        return

    line = char
    while len(line) < length:
        line += char
    print(line+'\n')

def header(text, char='_', length=100):
    if not CODECELL_OUTPUT:
        return

    padding = [f'{char}| ', f' |{char}']
    line = f'{char}| {text} |{char}'
    while len(line) < length:
        padding = [char+padding[0], padding[1]+char]
        line = padding[0] + text + padding[1]

    print(line)

## 5. Data preparation
Loads the live-state workbook, normalizes input columns, and builds the AP, location, labor, move-time, and paint-schedule frames consumed by the scheduler.

### AP, Location, and Labor Data
`load_live_state` reads `data/staged/FGI_liveState.xlsx` and returns the three input frames plus the staffing and CPJ dictionaries.

In [8]:
# build dataframes from FGI_LIVESTATE

def load_live_state(path='data/staged/FGI_liveState.xlsx'):
    global FGI_STAFFING_BYSHIFT, FGI_CPJ

    required_columns = {
        'ap_data': [
            'LN', 'FA RO', 'FA RO to B1R', 'Total Counters', 'TankStatus', 'Ceilings',
            'Initial Tests Run', 'BTG_tot', 'BTG_p0', 'BTG_p1', 'BTG_p2', 'BTG_p3',
            'BTG_engines', 'BTG_doors', 'BTG_test', 'P3_Engine Hang',
            'P3_Flight Controls', 'P3_Gear Swing', 'P3_Medium Pressure Test',
            'P3_Oil On', 'P3_Power On', 'P3_Engine_Type',
            'P3_Milestone_Completion_Percentage', 'shakes_complete', 'shakes_req'
        ],
        'location_data': [
            'Location', 'Future State Priority', 'Date Online', 'Owner',
            'tooling_jacking', 'tooling_wings', 'tooling_tankClosure',
            'centerline_dependencies', 'obstructions', 'notes'
        ],
        'labor_data': ['category', 'shift', 'team', 'value']
    }

    numeric_ap_columns = [
        'LN', 'FA RO to B1R', 'Total Counters', 'TankStatus', 'Ceilings',
        'Initial Tests Run', 'BTG_tot', 'BTG_p0', 'BTG_p1', 'BTG_p2', 'BTG_p3',
        'BTG_engines', 'BTG_doors', 'BTG_test', 'P3_Engine Hang',
        'P3_Flight Controls', 'P3_Gear Swing', 'P3_Medium Pressure Test',
        'P3_Oil On', 'P3_Power On', 'P3_Milestone_Completion_Percentage',
        'shakes_complete', 'shakes_req'
    ]

    tooling_columns = ['tooling_jacking', 'tooling_wings', 'tooling_tankClosure']

    def _normalize_bool(value):
        if pd.isna(value):
            return False
        if isinstance(value, (bool, np.bool_)):
            return bool(value)
        if isinstance(value, (int, float)):
            return value != 0

        value_str = str(value).strip().lower()
        if value_str in {'true', 't', 'yes', 'y', '1'}:
            return True
        if value_str in {'false', 'f', 'no', 'n', '0', ''}:
            return False

        raise ValueError(f'Invalid boolean value in location_data: {value}')

    path = Path(path)
    path_candidates = [path]
    if not path.is_absolute():
        path_candidates.extend([rootpath / path, Path.cwd() / path, Path.cwd().parent / path])
    else:
        path_candidates.append(Path.cwd().parent / 'data' / 'staged' / path.name)
    path = next((candidate for candidate in path_candidates if candidate.exists()), path)

    if not path.exists():
        raise ValueError(f'Live state workbook not found: {path}')

    try:
        workbook = pd.ExcelFile(path, engine='openpyxl')
    except FileNotFoundError as exc:
        raise ValueError(f'Live state workbook not found: {path}') from exc
    except Exception as exc:
        raise ValueError(f'Unable to open live state workbook: {path}') from exc

    missing_sheets = [sheet for sheet in required_columns if sheet not in workbook.sheet_names]
    if missing_sheets:
        raise ValueError(
            f"Missing required sheet(s) in {path}: {', '.join(missing_sheets)}"
        )

    ap_df = pd.read_excel(workbook, sheet_name='ap_data')
    location_df = pd.read_excel(workbook, sheet_name='location_data')
    labor_df = pd.read_excel(workbook, sheet_name='labor_data')

    frames = {
        'ap_data': ap_df,
        'location_data': location_df,
        'labor_data': labor_df
    }

    for sheet_name, expected_columns in required_columns.items():
        missing_columns = [col for col in expected_columns if col not in frames[sheet_name].columns]
        if missing_columns:
            raise ValueError(
                f"Missing required column(s) in sheet '{sheet_name}': {', '.join(missing_columns)}"
            )

    ap_df = ap_df.copy()
    ap_df['FA RO'] = pd.to_datetime(ap_df['FA RO'], errors='coerce')
    for column in numeric_ap_columns:
        ap_df[column] = pd.to_numeric(ap_df[column], errors='coerce')

    if ap_df['LN'].notna().all():
        ap_df['LN'] = ap_df['LN'].astype('Int64')

    location_df = location_df.copy()
    location_df['Location'] = location_df['Location'].astype('string').str.strip()
    location_df = location_df[location_df['Location'].notna() & location_df['Location'].ne('')].reset_index(drop=True)
    for column in tooling_columns:
        location_df[column] = location_df[column].map(_normalize_bool).astype(bool)

    labor_df = labor_df.copy()
    labor_df['category'] = labor_df['category'].astype('string').str.strip()
    labor_df['team'] = labor_df['team'].astype('string').str.strip()
    labor_df['shift'] = pd.to_numeric(labor_df['shift'], errors='coerce').astype('Int64')
    labor_df['value'] = pd.to_numeric(labor_df['value'], errors='coerce')

    staffing_rows = labor_df[labor_df['category'].eq('FGI_STAFFING_BYSHIFT')].copy()
    cpj_rows = labor_df[labor_df['category'].eq('FGI_CPJ')].copy()

    if staffing_rows.empty:
        raise ValueError("Missing FGI_STAFFING_BYSHIFT rows in sheet 'labor_data'")
    if cpj_rows.empty:
        raise ValueError("Missing FGI_CPJ rows in sheet 'labor_data'")

    if staffing_rows[['shift', 'team', 'value']].isna().any().any():
        raise ValueError("FGI_STAFFING_BYSHIFT rows must include shift, team, and value")
    if cpj_rows[['team', 'value']].isna().any().any():
        raise ValueError("FGI_CPJ rows must include team and value")

    FGI_STAFFING_BYSHIFT = {
        int(shift): group.set_index('team')['value'].astype(float).to_dict()
        for shift, group in staffing_rows.groupby('shift')
    }
    FGI_CPJ = cpj_rows.set_index('team')['value'].astype(float).to_dict()

    return ap_df, location_df, labor_df

ap_df, location_df, labor_df = load_live_state()


### Move and Paint Schedule Imports
Move-time matrix from `data/staged/move_times.xlsx` and the paint-bay assignment schedule from `data/staged/paint_schedule.xlsx`.

In [9]:
def load_move_times(filepath=filepaths['Move Times']):
    df = pd.read_excel(filepath, sheet_name='location_move_times', index_col=0)
    # set location names to strings
    df.index = df.index.astype(str)
    df.columns = df.columns.astype(str)

    df = df.apply(pd.to_numeric, errors='coerce')
    move_times = df.to_dict(orient='index')
    return move_times

def add_move_times(fgi, move_times):
    # ensure all locations exist
    for from_loc, to_dict in move_times.items():
        if from_loc not in fgi.Locations:
            fgi.add_Location(
                from_loc,
                Location(priority=0, dateOnline='Now', name=from_loc)
            )

        for to_loc in to_dict:
            if to_loc not in fgi.Locations:
                fgi.add_Location(
                    to_loc,
                    Location(priority=0, dateOnline='Now', name=to_loc)
                )

    # assign move times directly
    for from_loc, to_dict in move_times.items():
        loc_obj = fgi.Locations[from_loc]
        for to_loc, time in to_dict.items():
            loc_obj.set_time_to(to_loc, time)

    return fgi

In [10]:
move_times = load_move_times()

In [11]:
def load_paint_schedule(filepath=filepaths['Paint Schedule'], sheet_name='Historical'):

    paint_df = pd.read_excel(filepath, sheet_name=sheet_name, header=2, engine='openpyxl')
    paint_df = paint_df[['Date', 'BSC1', 'BSC2']].copy()

    paint_df['Date'] = pd.to_datetime(paint_df['Date'], errors='coerce').dt.normalize()
    paint_df = paint_df[paint_df['Date'].notna()].reset_index(drop=True)

    paint_schedule = {}

    for _, row in paint_df.iterrows():
        date = row['Date']
        paint_schedule[date] = {}

        for bay in ['BSC1', 'BSC2']:
            if pd.isna(row[bay]):
                paint_schedule[date][bay] = None
            else:
                paint_schedule[date][bay] = str(int(row[bay]))

    return paint_schedule


All input frames are now in memory. The next sections define the scheduler classes that consume them.

## 6. Scheduler classes
Class definitions for the scheduler: `AP`, `Location`, `FGI` (main scheduling class), and `FGITrace` (run-time event log used by the workbook export).

Active model rules captured by these classes:
- FA-owned locations are not placeable (`Location.canPlace()` returns `False` when `owner == 'FA'`).
- A1–A10 are the preferred DC/exit staging stalls; D1/D2 are legal DC overflow but are not explicitly required exit destinations.
- Paint scheduling has priority over compass scheduling in `reorder_move_queue`.
- Compass calibration completes only at the head of the compass queue, with CR3 occupancy plus CR1/CR2 clear plus one elapsed simulated workday.
- An AP entering BSC1/BSC2 starts paint timing; on the next move out of BSC1/BSC2 it is marked `painted = True`.
- Delivery requires intentional exit routing → physical arrival at a DC stall → CT/output recording → end-of-day pending-exit cleanup.

In [12]:
class AP:
    # =========================================================================
    # AP CLASS
    # =========================================================================
    # Aircraft-program object. Owns FA rollout state, FGI workload, scheduling
    # state, and production status flags. Instances are constructed once from
    # the live-state ap_df and reset between runs via reset_state().

    def __init__(self, LN, faro, toB1R, counters, btg=None, tankClosed=False,
                 ceilings=0, testsRun=0, p3_milestones=None, shakes=None):

        # --- FA rollout attributes (from liveState) ---
        self.LN = LN
        self.faro = faro
        self.toB1R = toB1R
        self.counters = counters
        self.btg = btg if btg is not None else {
            "tot": 0, "p0": 0, "p1": 0, "p2": 0, "p3": 0,
            "engines": 0, "doors": 0, "test": 0
        }
        # Imported from liveState; reserved for tank-closure / shake / P3 work.
        self.tankClosed = tankClosed
        self.ceilings = ceilings
        self.testsRun = testsRun
        self.p3milestones = p3_milestones if p3_milestones is not None else {'tot': 6, 'complete': 0}
        self.shakes = shakes if shakes is not None else {'complete': 0, 'req': 4}

        # --- Scheduling attributes ---
        self.schedule = {}
        self.moveReq = False
        self.path = []
        self.Location = None
        self.destination = None
        self.compassStartDate = None
        self.exitPending = False

        # --- FGI workload ---
        self.taskState = 'FA'
        self.FGI_btg = get_FGI_BTG(self.btg)

        # --- Production status flags ---
        # compassCalibrated and painted flip from move_ap / schedule_compass_moves.
        # structure/systems/declam/test flip from update_labor_status() based on remaining BTG.
        self.status = {
            'compassCalibrated': False,
            'painted': False,
            'structure': self.FGI_btg['structure'] <= 0,
            'systems': self.FGI_btg['systems'] <= 0,
            'declam': self.FGI_btg['declam'] <= 0,
            'test': self.FGI_btg['test'] <= 0
        }
        self.movePriority = 'normal'

        # --- Initial state snapshot (used by reset_state) ---
        self.initial_toB1R = toB1R
        self.initial_btg = copy.deepcopy(self.btg)
        self.initial_FGI_btg = copy.deepcopy(self.FGI_btg)
        self.initial_status = copy.deepcopy(self.status)

    # -------------------------------------------------------------------------
    # Get methods
    # -------------------------------------------------------------------------
    def get_LN(self): return str(self.LN)
    def get_FAROdate(self): return pd.to_datetime(self.faro)
    def get_daystoB1R(self): return pd.Timedelta(days=self.toB1R)
    def get_fgi_btg(self, team):
        return pd.to_numeric(self.FGI_btg[team]) if team in self.FGI_btg.keys() else False

    # -------------------------------------------------------------------------
    # Scheduling methods
    # -------------------------------------------------------------------------
    def reset_state(self):
        # Restore FA / FGI workload state.
        self.toB1R = self.initial_toB1R
        self.btg = copy.deepcopy(self.initial_btg)
        self.FGI_btg = copy.deepcopy(self.initial_FGI_btg)
        self.status = copy.deepcopy(self.initial_status)

        # Restore scheduling state.
        self.schedule = {}
        self.moveReq = False
        self.path = []
        self.Location = None
        self.taskState = 'FA'
        self.destination = None
        self.exitPending = False
        self.compassStartDate = None

    def update_labor_status(self):
        # Refresh FGI team completion flags from remaining FGI BTG.
        self.status['structure'] = self.get_fgi_btg('structure') <= 0
        self.status['systems'] = self.get_fgi_btg('systems') <= 0
        self.status['declam'] = self.get_fgi_btg('declam') <= 0
        self.status['test'] = self.get_fgi_btg('test') <= 0
        return self.status

    def is_exit_ready(self):
        # NOTE: refreshes labor status from current BTG before evaluating.
        # Returns True only when all production gates are clear and no move is pending.
        self.update_labor_status()

        return (
            self.status['compassCalibrated']
            and self.status['painted']
            and self.status['structure']
            and self.status['systems']
            and self.status['declam']
            and self.status['test']
            and not self.isMoveReq()
            and self.destination is None
        )

    def set_taskState(self, state):
        AP_TASK_STATES = ['FA', 'RO', 'idle', 'paint', 'compass', 'shake',
                          'btg_completion', 'tankClosure', 'toDC', 'exit', 'delivered']
        if state in AP_TASK_STATES:
            self.taskState = state
        else:
            if CODECELL_OUTPUT:
                header('INVALID METHOD CALL')
                print(f'LN: {self.LN} taskState set to {state}')
                line()
            return False

    # -------------------------------------------------------------------------
    # Move methods
    # -------------------------------------------------------------------------
    def get_move_candidates(self, fgi, allow_temp_locations=False):
        origin = self.Location
        candidates = []

        for loc_name, loc in fgi.Locations.items():
            if origin is not None and loc.name == origin.name:
                continue

            if loc.is_temp and not allow_temp_locations:
                continue

            if not loc.canPlace():
                continue

            if origin is None:
                move_time = 0
            else:
                move_time = origin.time_to.get(loc_name, np.inf)

                if not np.isfinite(move_time):
                    continue

            candidates.append({
                'destination': loc,
                'destination_name': loc.name,
                'priority': loc.priority,
                'move_time': move_time
            })

        return sorted(candidates, key=lambda c: (c['priority'], c['move_time']))

    def isMoveReq(self): return self.moveReq

    def requireMove(self, destination=None):
        # Internal AP-side state mutation. Callers must go through FGI.request_move()
        # so the move queue stays in sync with ap.moveReq.
        self.moveReq = True
        self.destination = destination
        if CODECELL_OUTPUT:
            header('MOVE REQUIRED')
            current_loc = self.Location.name if self.Location is not None else None
            print(f'LN: {self.LN} requires move from {current_loc}\nTask State: {self.taskState}')
            line()

    def get_move_rank(self, fgi):
        # Tuple ranks: (tier, destination priority, move time).
        # Tier 0: mandatory destination already set (paint, compass, exit).
        # Tier 1: AP picks its best candidate location.
        # Tier 2: no feasible destination right now.
        if self.destination is not None:
            return (0, 0, 0)

        candidates = self.get_move_candidates(fgi)
        if not candidates:
            return (2, 999, 999)

        top = candidates[0]
        return (1, top['priority'], top['move_time'])

    # -------------------------------------------------------------------------
    # BTG / task completion
    # -------------------------------------------------------------------------
    def complete_BTG(self, category, btg_budget=0, byLabor=False):
        # Complete BTG using either a direct BTG category or an FGI labor bucket.
        # Returns (btg_consumed, ap_complete_for_that_bucket).
        if btg_budget is None or btg_budget <= 0:
            if CODECELL_OUTPUT:
                header('METHOD ERROR: complete_BTG')
                print('tried to complete BTG with no available BTG budget')
                print(f'LN: {self.LN} | category: {category} | btg_budget: {btg_budget}')
                line()
            return 0, False

        if not byLabor and category not in self.btg:
            if CODECELL_OUTPUT:
                header('METHOD ERROR: complete_BTG')
                print(f'invalid BTG category when byLabor=False: {category}')
                print(f'valid categories: {list(self.btg.keys())}')
                line()
            return 0, False

        if byLabor and category not in self.FGI_btg:
            if CODECELL_OUTPUT:
                header('METHOD ERROR: complete_BTG')
                print(f'invalid labor category when byLabor=True: {category}')
                print(f'valid categories: {list(self.FGI_btg.keys())}')
                line()
            return 0, False

        # --- Direct BTG category completion ---
        if not byLabor:
            available_btg = max(self.btg[category], 0)
            btg_consumed = min(available_btg, btg_budget)

            self.btg[category] = max(self.btg[category] - btg_consumed, 0)
            self.btg['tot'] = max(self.btg['tot'] - btg_consumed, 0)

            if category in FGI_TEAMS_BY_BTG_TYPE and FGI_TEAMS_BY_BTG_TYPE[category] is not None:
                for team in FGI_TEAMS_BY_BTG_TYPE[category]:
                    if team in self.FGI_btg:
                        self.FGI_btg[team] = max(self.FGI_btg[team] - btg_consumed, 0)

            ap_complete = self.btg[category] <= 0

        # --- Labor bucket completion ---
        else:
            available_btg = max(self.FGI_btg[category], 0)
            btg_consumed = min(available_btg, btg_budget)

            self.FGI_btg[category] = max(self.FGI_btg[category] - btg_consumed, 0)

            if category in BTG_TYPES_BY_LABOR:
                for btg_type in BTG_TYPES_BY_LABOR[category]:
                    if btg_type in self.btg:
                        self.btg[btg_type] = max(self.btg[btg_type] - btg_consumed, 0)

            self.btg['tot'] = max(self.btg['tot'] - btg_consumed, 0)

            ap_complete = self.FGI_btg[category] <= 0

        # Refresh team status flags after every BTG completion.
        self.update_labor_status()

        return btg_consumed, ap_complete

In [13]:
class Location:
    # =========================================================================
    # LOCATION CLASS
    # =========================================================================
    # Physical or temporary FGI/FA location. Tracks one assigned AP at a time,
    # carries tooling/centerline metadata, and exposes canPlace() which the
    # scheduler uses for every placement decision.

    def __init__(self, priority, dateOnline, name, owner=None, tooling=None, centerlines=None):
        self.priority = priority

        if dateOnline == 'Now':
            self.isOnline = True
        elif dateOnline == 'At R10':
            self.isOnline = NEW_FA_ONLINE
        else:
            self.isOnline = False

        self.name = name
        self.is_temp = str(self.name).startswith('N')
        self.owner = owner
        self.tooling = tooling if tooling is not None else {
            'jacking': False, 'wings': False, 'tankClosure': False
        }

        centerline_list = [
            centerline.strip()
            for centerline in str(centerlines).split(',')
            if centerline.strip() not in ['', 'None', 'nan', 'N/A']
        ]
        self.centerlines = centerline_list if len(centerline_list) > 0 else None

        self.schedule = {}
        self.time_to = {}
        self.AP = None

    def canUse(self, tool):
        if tool in self.tooling.keys():
            return self.tooling[tool]
        return False

    def isAvailable(self):
        return self.AP is None

    def canPlace(self):
        # FA-owned locations are not placeable through the scheduler;
        # discontinued locations are excluded from candidate generation.
        if self.owner == 'FA':
            return False
        if self.name in DISCONTINUED_LOCATIONS:
            return False
        return self.isOnline and self.isAvailable()

    def assign(self, AP, date=None):
        if self.AP is not None:
            if CODECELL_OUTPUT:
                header('ERROR', '~')
                print("AP assigned to unavailable location")
                line('~')
            return False

        self.AP = AP
        if date is not None:
            self.schedule[date] = AP.get_LN()
        return True

    def unassign(self, date=None):
        if CODECELL_OUTPUT and self.AP is not None:
            header('MOVE', ' - ')
            print(f'{self.AP.get_LN()} removed from location {self.name}')
            line(' - ')

        if date is not None:
            self.schedule[date] = None

        self.AP = None

    def clear_schedule(self):
        self.schedule = {}

    def set_time_to(self, other, move_time):
        self.time_to[other] = move_time

In [14]:
# =============================================================================
# FGI SCHEDULER CLASS
# =============================================================================
# Owns all APs and locations active in FGI, plus the move / FGI-task / labor
# queues. Drives the daily simulation: rollout, labor allocation, paint and
# compass scheduling, move-queue processing, exit routing, and end-of-day
# pending-exit cleanup.

class FGI:
    def __init__(self, labor, CPJ, paint_schedule=None):
        self.labor = labor
        self.CPJ = CPJ
        self.paint_schedule = paint_schedule if paint_schedule is not None else None
        self.APs = {}
        self.Locations = {}
        self.sortedLocations = {
            'FA': {},
            'FGI': {},
            'DC': {},
            'temp': {}
        }
        self.queues = {
            'move': [],
            'FGI task': {
                'compass': [],
                'paint': [],
                'shake': []
            },
            'labor': {
                'structure': [],
                'systems': [],
                'declam': [],
                'test': []
            }
        }
        self.schedule = {}
        self.chickenTracks = {}
        self.apStateRows = []
        self.deliveryRows = []
        self.shift = None
        self.btg_budget = {team: 0 for team in FGI_TEAMS}
        self.movetime_remaining = 0
        self.pendingExitLNs = []

    # -------------------------------------------------------------------------
    # AP HANDLING METHODS
    # -------------------------------------------------------------------------
    def add_ap(self, ap):
        LN = ap.get_LN()

        if LN in self.APs:
            if CODECELL_OUTPUT:
                header('AP ALREADY IN FGI', '~')
                print(f'LN {LN} was already active in FGI; add_ap skipped')
                line('~')
            return False

        self.APs[LN] = ap
        ap.set_taskState('RO')

        if LN not in self.queues['FGI task']['paint']:
            self.queues['FGI task']['paint'].append(LN)

        if LN not in self.queues['FGI task']['compass']:
            self.queues['FGI task']['compass'].append(LN)

        for team in self.queues['labor']:
            remaining_btg = ap.get_fgi_btg(team)

            if remaining_btg > 0:
                ap.status[team] = False
                if LN not in self.queues['labor'][team]:
                    self.queues['labor'][team].append(LN)
            else:
                ap.status[team] = True

        if CODECELL_OUTPUT:
            header('AP ADDED TO FGI')
            print(f'LN {LN} added to FGI')
            print(f"Paint queue length: {len(self.queues['FGI task']['paint'])}")
            print(f"Compass queue length: {len(self.queues['FGI task']['compass'])}")
            for team in self.queues['labor']:
                print(f"{team} queue length: {len(self.queues['labor'][team])}")
            line()

        return True
    
    def get_active_ap_status_df(self):
        columns = [
            'LN',
            'Location',
            'Task_State',
            'Move_Req',
            'Destination',
            'Queues',
            'Queue_Count',
            'FGI_structure',
            'FGI_systems',
            'FGI_declam',
            'FGI_test',
            'Compass_Complete',
            'Paint_Complete'
        ]

        rows = []

        for LN, AP in self.APs.items():
            queue_membership = self.get_queue_membership(LN)

            rows.append({
                'LN': LN,
                'Location': None if AP.Location is None else AP.Location.name,
                'Task_State': AP.taskState,
                'Move_Req': AP.isMoveReq(),
                'Destination': AP.destination,
                'Queues': ', '.join(queue_membership),
                'Queue_Count': len(queue_membership),
                'FGI_structure': AP.get_fgi_btg('structure'),
                'FGI_systems': AP.get_fgi_btg('systems'),
                'FGI_declam': AP.get_fgi_btg('declam'),
                'FGI_test': AP.get_fgi_btg('test'),
                'Compass_Complete': AP.status.get('compassCalibrated', False),
                'Paint_Complete': AP.status.get('painted', False)
            })

        return pd.DataFrame(rows, columns=columns)

    def get_queue_membership(self, LN):
        memberships = []

        if LN in self.queues['move']:
            memberships.append('move')

        for task, queue in self.queues['FGI task'].items():
            if LN in queue:
                memberships.append(task)

        for team, queue in self.queues['labor'].items():
            if LN in queue:
                memberships.append(team)

        if LN in self.pendingExitLNs:
            memberships.append('pending_exit')

        return memberships

    def refresh_ap_states(self):
        # Active labor LNs (head, plus second when head won't exhaust budget).
        btg_active = set()
        for team in FGI_TEAMS:
            q = self.queues['labor'][team]
            budget = self.btg_budget[team]
            if not q or budget <= 0:
                continue
            head = q[0]
            btg_active.add(head)
            if self.APs[head].get_fgi_btg(team) < budget and len(q) >= 2:
                btg_active.add(q[1])

        compass_q = self.queues['FGI task']['compass']
        compass_head = compass_q[0] if compass_q else None

        for LN, ap in self.APs.items():
            if ap.taskState in ('exit', 'delivered'):
                continue
            loc = ap.Location.name if ap.Location else None

            if loc in ('BSC1', 'BSC2'):
                ap.taskState = 'paint'
            elif LN == compass_head and loc == 'CR3':
                ap.taskState = 'compass'
            elif LN in btg_active:
                ap.taskState = 'btg_completion'
            else:
                ap.taskState = 'idle'

    # -------------------------------------------------------------------------
    # LOCATION HANDLING METHODS
    # -------------------------------------------------------------------------
    def add_Location(self, name, location_obj):
        self.Locations[name] = location_obj

        # --- SORT LOCATION BY FUNCTIONAL GROUP ---
        # Group locations for later move selection, exit logic, and reporting.
        if str(name).startswith('N'):
            self.sortedLocations['temp'][name] = location_obj

        elif location_obj.owner == 'DC':
            self.sortedLocations['DC'][name] = location_obj

        elif str(name).startswith('P'):
            self.sortedLocations['FA'][name] = location_obj

        else:
            self.sortedLocations['FGI'][name] = location_obj

        self.chickenTracks[name] = None

        if CODECELL_OUTPUT:
            line()
            print(f'Location {name} added to FGI')
            print(f'Current locations in FGI: {list(self.Locations.keys())}')
            line()

    # -------------------------------------------------------------------------
    # LABOR HANDLING METHODS
    # -------------------------------------------------------------------------
        
    def assign_labor(self, team, date=None):

        btg_completed = 0
        worked_lns = []
        queue = self.queues['labor'][team]

        while self.btg_budget[team] > 0 and len(queue) > 0:
            LN = queue[0]
            ap = self.APs[LN]

            btg_consumed, ap_complete = ap.complete_BTG(
                team,
                btg_budget=self.btg_budget[team],
                byLabor=True
            )

            self.btg_budget[team] -= btg_consumed
            btg_completed += btg_consumed

            if btg_consumed > 0:
                worked_lns.append(LN)

                if hasattr(self, 'trace'):
                    self.trace.record_btg(date, LN, team, btg_consumed)

            if ap_complete:
                queue.pop(0)
            else:
                break

        btg_remaining = self.btg_budget[team]

        if hasattr(self, 'trace'):
            self.trace.record_labor(date, team, worked_lns)

        return worked_lns, btg_completed, btg_remaining
    
    # -------------------------------------------------------------------------
    # SCHEDULING / STATUS UPDATE METHODS
    # -------------------------------------------------------------------------
    def set_shift(self, shift):
        if shift not in [0, 1, 2, 3]:
            if CODECELL_OUTPUT:
                header('INVALID SHIFT')
                print(f'Shift {shift} is not valid. Shift must be 0, 1, 2, or 3.')
                line()
        
        if shift in [1, 2]:
            self.shift_labor = self.labor[shift]
            self.shift = shift
            self.btg_budget = {team: pd.to_numeric(self.shift_labor[team] / self.CPJ[team]) for team in FGI_TEAMS}
        
        if shift == 3:
            self.movetime_remaining = 8
            self.shift = shift

    def get_daily_status_df(self):
        if len(self.apStateRows) == 0:
            return pd.DataFrame(columns=[
                'Date',
                'LN',
                'Location',
                'FGI_tot',
                'structure',
                'systems',
                'declam',
                'test',
                'moveReq'
            ])

        return pd.DataFrame(self.apStateRows)
    
    def schedule_paint_moves(self, today):
        # --- SCHEDULED MOVES FOR PAINT ---
        # Paint scheduling creates move requests only. Paint completion occurs when AP leaves BSC.

        scheduled = []
        displaced = []

        tomorrow = today + pd.Timedelta(days=1)

        if self.paint_schedule is None:
            return {
                'scheduled': scheduled,
                'displaced': displaced
            }

        if tomorrow not in self.paint_schedule:
            return {
                'scheduled': scheduled,
                'displaced': displaced
            }

        for bay_name, scheduled_LN in self.paint_schedule[tomorrow].items():

            if bay_name not in self.Locations:
                continue

            bay = self.Locations[bay_name]
            current_ap = bay.AP
            current_LN = None if current_ap is None else current_ap.get_LN()

            # Move current bay occupant out if a different AP is scheduled for that bay.
            if current_LN is not None and current_LN != scheduled_LN:
                current_ap.taskState = 'btg_completion'
                self.request_move(
                    current_LN,
                    priority='clear_bay',
                    override=True
                )
                displaced.append(current_LN)

            # Move tomorrow's scheduled AP into the bay.
            if scheduled_LN is not None and scheduled_LN in self.APs:
                scheduled_ap = self.APs[scheduled_LN]
                scheduled_ap.taskState = 'paint'
                self.request_move(
                    scheduled_LN,
                    destination=bay_name,
                    priority='paint',
                    override=True
                )
                scheduled.append(scheduled_LN)

        return {
            'scheduled': scheduled,
            'displaced': displaced
        }
    
    def schedule_compass_moves(self, today):
        # --- SCHEDULED MOVES FOR COMPASS ---
        # Compass queue order is source-of-truth; only the head can complete.

        completed = None
        requested = None
        blocked_by = None

        compass_queue = self.queues['FGI task']['compass']
        if len(compass_queue) == 0:
            return {
                'completed': completed,
                'requested': requested,
                'blocked_by': blocked_by
            }

        head_LN = compass_queue[0]
        cr1 = self.Locations.get('CR1')
        cr2 = self.Locations.get('CR2')
        cr3 = self.Locations.get('CR3')

        cr1_clear = cr1 is None or cr1.AP is None
        cr2_clear = cr2 is None or cr2.AP is None

        if cr3 is None:
            blocked_by = 'missing_CR3'

        elif cr3.AP is not None:
            cr3_LN = cr3.AP.get_LN()

            if cr3_LN == head_LN and head_LN in self.APs:
                ap = self.APs[head_LN]
                has_start = ap.compassStartDate is not None
                waited_one_workday = (
                    has_start
                    and pd.Timestamp(today).normalize() > pd.Timestamp(ap.compassStartDate).normalize()
                )

                if cr1_clear and cr2_clear and waited_one_workday:
                    ap.status['compassCalibrated'] = True
                    ap.taskState = 'btg_completion'
                    self.dequeue('compass', head_LN)
                    self.request_move(
                        head_LN,
                        priority='compass_clear',
                        override=False
                    )
                    completed = head_LN
                else:
                    blocked_by = head_LN

            else:
                blocked_by = cr3_LN
                if cr3_LN in self.APs:
                    self.request_move(
                        cr3_LN,
                        priority='compass_clear',
                        override=False
                    )

        if cr3 is not None and cr3.AP is None and head_LN in self.APs:
            next_ap = self.APs[head_LN]
            next_ap.taskState = 'compass'
            if self.request_move(
                head_LN,
                destination='CR3',
                priority='compass',
                override=False
            ):
                requested = head_LN

        return {
            'completed': completed,
            'requested': requested,
            'blocked_by': blocked_by
        }


    def dequeue(self, queue_name, LN):
        removed = False

        if queue_name == 'all':
            removed_any = False

            for q_name in ['move']:
                removed_any = self.dequeue(q_name, LN) or removed_any

            for q_name in self.queues['FGI task']:
                removed_any = self.dequeue(q_name, LN) or removed_any

            for q_name in self.queues['labor']:
                removed_any = self.dequeue(q_name, LN) or removed_any

            return removed_any

        if queue_name == 'move':
            queue = self.queues['move']

            while LN in queue:
                queue.remove(LN)
                removed = True

            return removed

        if queue_name in self.queues['FGI task']:
            queue = self.queues['FGI task'][queue_name]

            while LN in queue:
                queue.remove(LN)
                removed = True

            return removed

        if queue_name in self.queues['labor']:
            queue = self.queues['labor'][queue_name]

            while LN in queue:
                queue.remove(LN)
                removed = True

            return removed

        return False

    # -------------------------------------------------------------------------
    # MOVE METHODS
    # -------------------------------------------------------------------------
    
    def get_req_centerline_moves(self, origin):
        blockers = []

        if origin is None:
            return None

        if origin.centerlines is None:
            return None

        for blocker_loc_name in origin.centerlines:
            blocker_loc_name = str(blocker_loc_name).strip()

            if blocker_loc_name not in self.Locations:
                continue

            blocker_loc = self.Locations[blocker_loc_name]

            if blocker_loc.AP is not None:
                blockers.append(blocker_loc.AP)

        if len(blockers) == 0:
            return None

        return blockers

    def centerline_blockers(self, blockers):
        centerline_nodes = ['N32', 'N31', 'N30', 'N29', 'N28']
        centerline_move_time = 0
        if blockers is None:
            return 0
        for blocker in blockers:
            centerline_move_time += blocker.Location.time_to[centerline_nodes.pop(0)] 
        centerline_move_time *= 2 # account for move out and return
        return centerline_move_time

    def move_ap(self, LN, destination, date=None):
        # --- FEASIBILITY CHECK ---
        if destination is None:
            return False, 0

        if destination.canPlace() is False:
            return False, 0
                
        ap = self.APs[LN]
        origin = ap.Location

        if origin is not None and origin.name == 'CR3' and not ap.status.get('compassCalibrated', False):
            ap.compassStartDate = None

        move_time = 0

        # --- APPLY CENTERLINES ---
        req_centerline_moves = self.get_req_centerline_moves(origin)
        
        if req_centerline_moves is not None:
            move_time += self.centerline_blockers(req_centerline_moves)
        
        # --- REMOVE FROM CURRENT LOCATION ---
        if origin is not None:
            direct_move_time = origin.time_to.get(destination.name, np.inf)
            if not np.isfinite(direct_move_time):
                if CODECELL_OUTPUT:
                    header('METHOD ERROR: move_ap', '~')
                    print(f'No move time defined from {origin.name} to {destination.name}')
                    line('~')
                return False, move_time
            move_time += direct_move_time

            if origin.name in ['BSC1', 'BSC2']:
                ap.status['painted'] = True
                self.dequeue('paint', LN)

        else:
            move_time = 0
            
        # --- MOVE TO DESTINATION ---
        if move_time <= self.movetime_remaining:
            if origin is not None:
                ap.path.append(origin.name)
                origin.unassign(date=date)
                self.chickenTracks[origin.name] = None


            destination.assign(ap, date=date)
            if destination.name in ['BSC1', 'BSC2']:
                ap.paintStartDate = pd.Timestamp(date).normalize() if date is not None else None
                ap.taskState = 'paint'
            ap.Location = destination
            self.chickenTracks[destination.name] = LN

            if destination.name == 'CR3' and self.queues['FGI task']['compass'] and self.queues['FGI task']['compass'][0] == LN:
                ap.compassStartDate = pd.Timestamp(date).normalize() if date is not None else None

            self.movetime_remaining -= move_time
        else:
            if CODECELL_OUTPUT:
                header('METHOD ERROR: move_ap', '~')
                print(f'Move time from {origin.name if origin else "None"} to {destination.name} is {move_time} hours, which exceeds the 8 hour limit')
                line('~')
            return False, move_time

        # --- MOVE COMPLETION STATE ---
        # Clear move request state only after the target AP has moved.
        ap.moveReq = False
        ap.destination = None
        ap.movePriority = 'normal'

        if hasattr(self, 'trace'):
            self.trace.record_move(date, LN, destination.name)

        return True, move_time
    
    def request_move(self, LN, destination=None, priority='normal', override=False):
        # --- ENQUEUE A MOVE REQUEST ---
        if LN not in self.APs:
            return False

        ap = self.APs[LN]

        if destination is not None and ap.Location is not None and ap.Location.name == destination:
            ap.moveReq = False
            ap.destination = None
            ap.movePriority = 'normal'
            return False

        if ap.isMoveReq() and not override:
            return False

        ap.requireMove(destination=destination)
        ap.movePriority = priority

        if LN not in self.queues['move']:
            self.queues['move'].append(LN)

        return True
    
    def execute_move_requests(self, date=None):
        # --- EXECUTE MOVES IN QUEUE ORDER ---
        # self.queues['move'] is the source of truth for movement order.
        results = []
        progress = True

        while progress:
            progress = False

            if self.movetime_remaining <= 0:
                break

            i = 0

            while i < len(self.queues['move']):
                if self.movetime_remaining <= 0:
                    break

                LN = self.queues['move'][i]

                if LN not in self.APs:
                    self.queues['move'].pop(i)
                    continue

                ap = self.APs[LN]

                if not ap.isMoveReq():
                    self.queues['move'].pop(i)
                    continue

                if ap.destination is not None:
                    destination = self.Locations.get(ap.destination)
                else:
                    candidates = ap.get_move_candidates(self)
                    destination = candidates[0]['destination'] if candidates else None

                if destination is None:
                    results.append({
                        'LN': LN,
                        'destination': None,
                        'moved': False,
                        'move_time': 0,
                        'reason': 'no_destination'
                    })
                    i += 1
                    continue

                moved, move_time = self.move_ap(LN, destination, date=date)

                results.append({
                    'LN': LN,
                    'destination': destination.name,
                    'moved': moved,
                    'move_time': move_time,
                    'reason': 'moved' if moved else 'move_failed'
                })

                if moved:
                    self.queues['move'].pop(i)
                    progress = True
                else:
                    i += 1

            if progress and self.movetime_remaining > 0:
                self.assign_exit_destinations()
                self.reorder_move_queue()

        return results
    
    def complete_AP(self, LN, date=None):
        if LN not in self.APs:
            return False

        AP = self.APs[LN]
        AP.exitPending = False
        AP.taskState = 'delivered'

        actual_exit_date = pd.Timestamp(date) if date is not None else None
        fa_ro_date = AP.get_FAROdate()

        planned_b1r_date = None
        days_late = None
        time_in_system_days = None

        if hasattr(AP, 'initial_toB1R'):
            planned_b1r_date = fa_ro_date + pd.Timedelta(days=AP.initial_toB1R)

        if actual_exit_date is not None:
            time_in_system_days = (actual_exit_date - fa_ro_date).days

            if planned_b1r_date is not None:
                days_late = (actual_exit_date - planned_b1r_date).days

        self.deliveryRows.append({
            'LN': LN,
            'FA_RO_Date': fa_ro_date,
            'Planned_B1R_Date': planned_b1r_date,
            'Actual_Exit_Date': actual_exit_date,
            'Time_In_System_Days': time_in_system_days,
            'Days_Late': days_late,
            'Final_Location': None if AP.Location is None else AP.Location.name
        })

        self.dequeue('all', LN)

        if AP.Location is not None:
            loc_name = AP.Location.name
            AP.Location.unassign()
            self.chickenTracks[loc_name] = None

        AP.Location = None
        self.APs.pop(LN)

        if CODECELL_OUTPUT:
            header('AP DELIVERED')
            print(f'LN {LN} has been delivered and removed from FGI')
            print(f'Time in system: {time_in_system_days}')
            print(f'Days late: {days_late}')
            line()

        return True
 
    def get_open_dc_stall(self, exclude=None):
        exclude = exclude or set()
        dc_locations = self.sortedLocations.get('DC', {})

        candidates = []

        for stall_name, stall in dc_locations.items():
            if stall_name in exclude:
                continue

            if stall is None:
                continue

            if not stall.canPlace():
                continue

            if any(ap.destination == stall_name for ap in self.APs.values()):
                continue

            candidates.append(stall)

        if len(candidates) == 0:
            return None

        def dc_rank(loc):
            # Prefer A1-A10. Allow D1/D2 only as overflow.
            is_d_location = str(loc.name).startswith('D')
            return (
                1 if is_d_location else 0,
                loc.priority
            )

        return sorted(candidates, key=dc_rank)[0]
    
    def reorder_move_queue(self):
        priority_rank = {
            'paint': 0,
            'clear_bay': 1,
            'exit': 2,
            'compass_clear': 3,
            'compass': 4,
            'normal': 5
        }

        def rank(LN):
            if LN not in self.APs:
                return (99, 99, np.inf)

            ap = self.APs[LN]
            priority = getattr(ap, 'movePriority', 'normal')
            priority_value = priority_rank.get(priority, 5)

            if ap.destination is not None:
                destination = self.Locations.get(ap.destination)

                if destination is None:
                    return (priority_value, 99, np.inf)

                origin = ap.Location
                move_time = 0 if origin is None else origin.time_to.get(destination.name, np.inf)

                return (priority_value, destination.priority, move_time)

            candidates = ap.get_move_candidates(self)
            if len(candidates) == 0:
                return (priority_value, 99, np.inf)

            best = candidates[0]

            return (priority_value, best['priority'], best['move_time'])

        self.queues['move'].sort(key=rank)
    
    # -------------------------------------------------------------------------
    # EXIT METHODS
    # -------------------------------------------------------------------------
    def assign_exit_destinations(self):
        # --- ASSIGN DC STALLS TO EXIT-READY APs ---
        # Reserves stalls inline so concurrent exit-ready APs don't collide.
        claimed = set()

        for LN, ap in self.APs.items():
            if not ap.is_exit_ready():
                continue

            if ap.Location is not None and ap.Location.name in self.sortedLocations['DC']:
                ap.taskState = 'exit'
                ap.exitPending = True
                continue

            if ap.destination is not None:
                continue

            stall = self.get_open_dc_stall(exclude=claimed)
            if stall is None:
                break

            ap.taskState = 'exit'
            ap.exitPending = True
            self.request_move(LN, destination=stall.name, priority='exit')
            claimed.add(stall.name)


    def mark_dc_arrivals_pending(self):
        # --- FLAG EXIT-ROUTED APs AT DC FOR END-OF-DAY EXIT ---
        marked = []

        for LN, ap in self.APs.items():
            if not getattr(ap, 'exitPending', False):
                continue

            if ap.Location is None:
                continue

            if ap.Location.name not in self.sortedLocations['DC']:
                continue

            if LN in self.pendingExitLNs:
                continue

            self.pendingExitLNs.append(LN)
            marked.append(LN)

        return marked


    def complete_pending_exits(self, date=None):
        # --- FINALIZE EXITS RECORDED THIS DAY ---
        # Called after record_day so today's CT row already shows the AP at DC.
        completed = []

        for LN in list(self.pendingExitLNs):
            if LN not in self.APs:
                continue

            if self.complete_AP(LN, date=date):
                completed.append(LN)

        self.pendingExitLNs = []
        return completed


    # -------------------------------------------------------------------------
    # KPI METHODS
    # -------------------------------------------------------------------------
    def record_day(self, date):
        date = pd.Timestamp(date)

        self.apStateRows = [
            row for row in self.apStateRows
            if pd.Timestamp(row['Date']) != date
        ]

        self.schedule[date] = {}

        for loc_name, loc in self.Locations.items():
            self.schedule[date][loc_name] = None if loc.AP is None else loc.AP.get_LN()

        for LN, ap in self.APs.items():
            self.apStateRows.append({
                'Date': date,
                'LN': LN,
                'Location': None if ap.Location is None else ap.Location.name,
                'FGI_tot': ap.get_fgi_btg('FGI_tot'),
                'structure': ap.get_fgi_btg('structure'),
                'systems': ap.get_fgi_btg('systems'),
                'declam': ap.get_fgi_btg('declam'),
                'test': ap.get_fgi_btg('test'),
                'moveReq': ap.isMoveReq()
            })
   
    def get_kpi_summary_df(self, trace=None):
        rows = []

        delivery_df = self.get_delivery_summary_df()
        daily_status_df = self.get_daily_status_df()

        active_status_df = pd.DataFrame([
            {
                'LN': LN,
                'Location': None if AP.Location is None else AP.Location.name,
                'Task_State': AP.taskState,
                'Move_Req': AP.isMoveReq(),
                'Destination': AP.destination,
                'FGI_structure': AP.get_fgi_btg('structure'),
                'FGI_systems': AP.get_fgi_btg('systems'),
                'FGI_declam': AP.get_fgi_btg('declam'),
                'FGI_test': AP.get_fgi_btg('test'),
                'Compass_Complete': AP.status.get('compassCalibrated', False),
                'Paint_Complete': AP.status.get('painted', False),
                'Exit_Ready': AP.is_exit_ready() if hasattr(AP, 'is_exit_ready') else False
            }
            for LN, AP in self.APs.items()
        ])

        delivered_count = len(delivery_df)
        active_count = len(active_status_df)

        avg_time_in_system = None
        if delivered_count > 0 and 'Time_In_System_Days' in delivery_df.columns:
            avg_time_in_system = delivery_df['Time_In_System_Days'].mean()

        avg_days_late = None
        if delivered_count > 0 and 'Days_Late' in delivery_df.columns:
            avg_days_late = delivery_df['Days_Late'].mean()

        rows.append({
            'KPI': 'Delivered AP Count',
            'Value': delivered_count,
            'Definition': 'Number of APs delivered through complete_AP()'
        })

        rows.append({
            'KPI': 'Active AP Count at Termination',
            'Value': active_count,
            'Definition': 'Number of APs still active in FGI at the time this KPI table was generated'
        })

        rows.append({
            'KPI': 'Average Days in System',
            'Value': avg_time_in_system,
            'Definition': 'Average Actual_Exit_Date - FA_RO_Date for delivered APs'
        })

        rows.append({
            'KPI': 'Average Days Late',
            'Value': avg_days_late,
            'Definition': 'Average Actual_Exit_Date - Planned_B1R_Date for delivered APs'
        })

        if len(active_status_df) > 0:
            rows.append({
                'KPI': 'Active Exit-Ready AP Count',
                'Value': int(active_status_df['Exit_Ready'].sum()),
                'Definition': 'Active APs whose AP.is_exit_ready() returned True'
            })

            rows.append({
                'KPI': 'Active APs With Pending Move Request',
                'Value': int(active_status_df['Move_Req'].sum()),
                'Definition': 'Active APs with moveReq currently set to True'
            })

            rows.append({
                'KPI': 'Active APs With Destination Assigned',
                'Value': int(active_status_df['Destination'].notna().sum()),
                'Definition': 'Active APs with a non-null destination field'
            })

            for status_col in ['Compass_Complete', 'Paint_Complete']:
                rows.append({
                    'KPI': f'Active APs - {status_col}',
                    'Value': int(active_status_df[status_col].sum()),
                    'Definition': f'Number of active APs where {status_col} is True'
                })

        if len(daily_status_df) > 0 and 'Date' in daily_status_df.columns and 'LN' in daily_status_df.columns:
            located_daily = daily_status_df.dropna(subset=['Location']) if 'Location' in daily_status_df.columns else pd.DataFrame()

            avg_located_aps = None
            if len(located_daily) > 0:
                avg_located_aps = located_daily.groupby('Date')['LN'].nunique().mean()

            rows.append({
                'KPI': 'Average Located APs Per Day',
                'Value': avg_located_aps,
                'Definition': 'Average number of APs assigned to a physical location per recorded day'
            })

        if trace is not None:
            try:
                trace_outputs = trace.to_dataframes()

                if len(trace_outputs) == 4:
                    chickentracks_df, labor_df, moves_df, btg_dfs = trace_outputs
                else:
                    chickentracks_df, labor_df, moves_df, btg_dfs = None, None, pd.DataFrame(), {}

                if moves_df is not None and len(moves_df) > 0:
                    move_count = moves_df.notna().sum().sum()
                else:
                    move_count = 0

                rows.append({
                    'KPI': 'Total Successful Moves Recorded',
                    'Value': move_count,
                    'Definition': 'Count of nonblank LN destination records in Moves Per Day trace output'
                })

                if isinstance(btg_dfs, dict):
                    for team, btg_df in btg_dfs.items():
                        if btg_df is None or len(btg_df) == 0:
                            avg_days_worked = None
                            total_btg = 0
                        else:
                            df = btg_df.copy().reset_index()
                            date_col = 'Date' if 'Date' in df.columns else df.columns[0]
                            value_cols = [c for c in df.columns if c != date_col]

                            if len(value_cols) == 0:
                                avg_days_worked = None
                                total_btg = 0
                            else:
                                long_df = df.melt(
                                    id_vars=[date_col],
                                    value_vars=value_cols,
                                    var_name='LN',
                                    value_name='BTG_Completed'
                                )

                                long_df = long_df[pd.notna(long_df['BTG_Completed'])]
                                long_df = long_df[long_df['BTG_Completed'] > 0]

                                if len(long_df) > 0:
                                    days_by_ln = long_df.groupby('LN')[date_col].nunique()
                                    avg_days_worked = days_by_ln.mean()
                                    total_btg = long_df['BTG_Completed'].sum()
                                else:
                                    avg_days_worked = None
                                    total_btg = 0

                        rows.append({
                            'KPI': f'Average Days Worked - {team}',
                            'Value': avg_days_worked,
                            'Definition': f'Average number of unique workdays with positive BTG completion per AP for {team}'
                        })

                        rows.append({
                            'KPI': f'Total BTG Completed - {team}',
                            'Value': total_btg,
                            'Definition': f'Total positive BTG completed by {team}'
                        })

            except Exception as e:
                rows.append({
                    'KPI': 'KPI Trace Error',
                    'Value': str(e),
                    'Definition': 'Trace-based KPI calculation failed'
                })

        return pd.DataFrame(rows)
    
    def get_delivery_summary_df(self):
        if len(self.deliveryRows) == 0:
            return pd.DataFrame(columns=[
                'LN',
                'FA_RO_Date',
                'Planned_B1R_Date',
                'Actual_Exit_Date',
                'Time_In_System_Days',
                'Days_Late',
                'Final_Location'
            ])

        return pd.DataFrame(self.deliveryRows)

    def get_team_kpi_df(self, trace=None):
        rows = []

        default_columns = [
            'Team',
            'AP_Count_Worked',
            'Total_BTG_Completed',
            'Average_Days_Worked_Per_AP',
            'Max_Days_Worked_On_One_AP',
            'Average_BTG_Per_Workday'
        ]

        if trace is None:
            return pd.DataFrame(columns=default_columns)

        try:
            trace_outputs = trace.to_dataframes()

            if len(trace_outputs) == 4:
                chickentracks_df, labor_df, moves_df, btg_dfs = trace_outputs
            else:
                return pd.DataFrame(columns=default_columns)

            if not isinstance(btg_dfs, dict):
                return pd.DataFrame(columns=default_columns)

            for team, btg_df in btg_dfs.items():
                if btg_df is None or len(btg_df) == 0:
                    rows.append({
                        'Team': team,
                        'AP_Count_Worked': 0,
                        'Total_BTG_Completed': 0,
                        'Average_Days_Worked_Per_AP': None,
                        'Max_Days_Worked_On_One_AP': None,
                        'Average_BTG_Per_Workday': None
                    })
                    continue

                df = btg_df.copy().reset_index()
                date_col = 'Date' if 'Date' in df.columns else df.columns[0]
                value_cols = [c for c in df.columns if c != date_col]

                if len(value_cols) == 0:
                    rows.append({
                        'Team': team,
                        'AP_Count_Worked': 0,
                        'Total_BTG_Completed': 0,
                        'Average_Days_Worked_Per_AP': None,
                        'Max_Days_Worked_On_One_AP': None,
                        'Average_BTG_Per_Workday': None
                    })
                    continue

                long_df = df.melt(
                    id_vars=[date_col],
                    value_vars=value_cols,
                    var_name='LN',
                    value_name='BTG_Completed'
                )

                long_df = long_df[pd.notna(long_df['BTG_Completed'])]
                long_df = long_df[long_df['BTG_Completed'] > 0]

                if len(long_df) == 0:
                    rows.append({
                        'Team': team,
                        'AP_Count_Worked': 0,
                        'Total_BTG_Completed': 0,
                        'Average_Days_Worked_Per_AP': None,
                        'Max_Days_Worked_On_One_AP': None,
                        'Average_BTG_Per_Workday': None
                    })
                    continue

                days_by_ln = long_df.groupby('LN')[date_col].nunique()
                btg_by_day = long_df.groupby(date_col)['BTG_Completed'].sum()

                rows.append({
                    'Team': team,
                    'AP_Count_Worked': long_df['LN'].nunique(),
                    'Total_BTG_Completed': long_df['BTG_Completed'].sum(),
                    'Average_Days_Worked_Per_AP': days_by_ln.mean(),
                    'Max_Days_Worked_On_One_AP': days_by_ln.max(),
                    'Average_BTG_Per_Workday': btg_by_day.mean()
                })

            return pd.DataFrame(rows, columns=default_columns)

        except Exception:
            return pd.DataFrame(columns=default_columns)

In [15]:
# =============================================================================
# FGI TRACE CLASS
# =============================================================================
# Run-time event log used by the workbook export. Records daily location
# occupancy, labor allocation, moves, and BTG completion. to_dataframes()
# returns the four sheets consumed by the export cell.

class FGITrace:
    def __init__(self):
        self.chickentracks = {}
        self.labor_allocation = {}
        self.moves = {}
        self.btg_completion = {}

    @staticmethod
    def _date_key(date):
        return pd.Timestamp(date).normalize()

    @staticmethod
    def _ln_key(LN):
        if LN is None or pd.isna(LN):
            return None

        if isinstance(LN, (int, np.integer)):
            return str(int(LN))

        if isinstance(LN, (float, np.floating)) and float(LN).is_integer():
            return str(int(LN))

        return str(LN)

    def record_day_start(self, date, fgi):
        date = self._date_key(date)

        self.chickentracks[date] = {
            loc_name: self._ln_key(loc.AP.get_LN()) if loc.AP is not None else None
            for loc_name, loc in fgi.Locations.items()
        }

    def record_labor(self, date, team, LNs):
        date = self._date_key(date)
        team = str(team)

        if date not in self.labor_allocation:
            self.labor_allocation[date] = {}

        if isinstance(LNs, (list, tuple, set, pd.Series, np.ndarray)):
            normalized_lns = [self._ln_key(LN) for LN in LNs]
            normalized_lns = [LN for LN in normalized_lns if LN is not None]
            self.labor_allocation[date][team] = ', '.join(normalized_lns)
        elif LNs is None:
            self.labor_allocation[date][team] = None
        else:
            self.labor_allocation[date][team] = self._ln_key(LNs)

    def record_move(self, date, LN, target_location):
        date = self._date_key(date)
        LN = self._ln_key(LN)

        if LN is None:
            return

        if date not in self.moves:
            self.moves[date] = {}

        self.moves[date][LN] = None if target_location is None else str(target_location)

    def record_btg(self, date, LN, skill, btg_completed):
        date = self._date_key(date)
        LN = self._ln_key(LN)
        skill = str(skill)
        btg_completed = pd.to_numeric(btg_completed, errors='coerce')

        if LN is None or pd.isna(btg_completed):
            return

        if skill not in self.btg_completion:
            self.btg_completion[skill] = {}

        if date not in self.btg_completion[skill]:
            self.btg_completion[skill][date] = {}

        current = self.btg_completion[skill][date].get(LN, 0)
        self.btg_completion[skill][date][LN] = current + float(btg_completed)

    @staticmethod
    def _sorted_trace_df(data):
        df = pd.DataFrame.from_dict(data, orient='index')
        df.index = pd.to_datetime(df.index)
        df = df.sort_index()
        df.index.name = 'Date'
        return df

    def to_dataframes(self):
        chickentracks_df = self._sorted_trace_df(self.chickentracks)

        labor_df = self._sorted_trace_df(self.labor_allocation)
        for team in FGI_TEAMS:
            if team not in labor_df.columns:
                labor_df[team] = None
        labor_df = labor_df[FGI_TEAMS]

        moves_df = self._sorted_trace_df(self.moves)

        btg_dfs = {}
        for team in FGI_TEAMS:
            data = self.btg_completion.get(team, {})
            btg_dfs[team] = self._sorted_trace_df(data)

        return chickentracks_df, labor_df, moves_df, btg_dfs


## 7. Object constructors
`init_APs` and `init_Locations` materialize the dataframes from section 5 into AP and Location instances.

In [16]:
def init_APs(ap_df):
    return {
        str(row['LN']): AP(
            LN=int(row['LN']),
            faro=row['FA RO'],
            toB1R=row['FA RO to B1R'],
            counters=row['Total Counters'],
            btg={
                'tot': row['BTG_tot'],
                'p0': row['BTG_p0'],
                'p1': row['BTG_p1'],
                'p2': row['BTG_p2'],
                'p3': row['BTG_p3'],
                'engines': row['BTG_engines'],
                'doors': row['BTG_doors'],
                'test': row['BTG_test']
            },
            tankClosed=row['TankStatus'],
            ceilings=row['Ceilings'],
            testsRun=row['Initial Tests Run'],
            p3_milestones={
                'Engine Hang': row['P3_Engine Hang'],
                'Flight Controls': row['P3_Flight Controls'],
                'Gear Swing': row['P3_Gear Swing'],
                'Medium Pressure Test': row['P3_Medium Pressure Test'],
                'Oil On': row['P3_Oil On'],
                'Power On': row['P3_Power On'],
                'Engine_Type': row['P3_Engine_Type'],
                'Milestone_Completion_Percentage': row['P3_Milestone_Completion_Percentage']
            },
            shakes={'complete': row['shakes_complete'], 'req': row['shakes_req']}
        )
        for _, row in ap_df.iterrows()
    }

def init_Locations(location_df):
    Locations = {}
    for _, row in location_df.iterrows():
        if not pd.isna(row['Location']):
            Locations[row['Location']] = Location(
                priority=row['Future State Priority'],
                dateOnline=row['Date Online'],
                name=row['Location'],
                owner=row['Owner'],
                tooling={
                    'jacking': row['tooling_jacking'],
                    'wings': row['tooling_wings'],
                    'tankClosure': row['tooling_tankClosure']
                },
                centerlines = row['centerline_dependencies']
            )
    return Locations

## 8. Model initialization and main scheduling loop
Builds the FGI scheduler from the input frames, then runs the day-by-day simulation: rollout from FA, three labor shifts, third-shift movement planning, queue execution, DC pending-exit marking, end-of-day refresh and record, and pending-exit completion.

In [17]:
# Initialize based on BSC input
APs = init_APs(ap_df)
Locations = init_Locations(location_df)


In [18]:
# =============================================================================
# MAIN SCHEDULING ALGORITHM
# =============================================================================
# Phases per simulated day:
#   1) terminate or enter forecasting mode
#   2) rollout APs from FA
#   3) shift 0   - state refresh / weekday gating
#   4) shifts 1+2 - labor completion
#   5) shift 3   - third-shift movement (paint, compass, exit-routing, queue execution)
#   6) end-of-day - refresh AP states, record day, complete pending exits
# scheduling/loop variables
START = pd.to_datetime(STARTDATE)
END = pd.to_datetime(ENDDATE)
dates = pd.date_range(START, END)
FORECAST_CAP_DAYS = 365
forecast_end = END + pd.Timedelta(days=FORECAST_CAP_DAYS)
FORECAST_UNTIL_COMPLETION = True

today = START
TERMINATION = False


for LN, ap in APs.items():
    ap.reset_state()

for loc_name, loc in Locations.items():
    loc.AP = None
    loc.schedule = {}

paint_schedule = load_paint_schedule(filepath=filepaths['Paint Schedule'], sheet_name='Historical')

# initialize FGI object and add APs and Locations
fgi = FGI(labor=FGI_STAFFING_BYSHIFT, CPJ=FGI_CPJ, paint_schedule=paint_schedule)
trace = FGITrace()
fgi.trace = trace

for loc_name, loc in Locations.items():
    fgi.add_Location(loc_name, loc)


add_move_times(fgi, move_times)



# ============================= MAIN SCHEDULING LOOP ============================
while TERMINATION == False:
    if CODECELL_OUTPUT:
        header(f'{today.strftime("%Y-%m-%d")}', '=')
    # --- TERMINATION CONDITIONS ---
    if today > END:
        if not FORECAST_UNTIL_COMPLETION:
            TERMINATION = True
            if CODECELL_OUTPUT:
                header('END DATE REACHED', '>')
                header('TERMINATION', '==>')
                line('>')
            break

        if len(fgi.APs) == 0:
            TERMINATION = True
            if CODECELL_OUTPUT:
                header('ALL APS DELIVERED', '>')
                header('TERMINATION', '==>')
                line('>')
            break

        if today > forecast_end:
            TERMINATION = True
            if CODECELL_OUTPUT:
                header('FORECAST CAP REACHED', '>')
                print(f'Active APs remaining: {list(fgi.APs.keys())}')
                print(fgi.get_active_ap_status_df())
                line('>')
            break

        if CODECELL_OUTPUT:
            header('FORECASTING MODE', '>')

    trace.record_day_start(today, fgi)
    # --- ROLLOUT APs FROM FA ---
    for LN, ap in APs.items():
        if LN not in fgi.APs and today == ap.get_FAROdate():
            fgi.add_ap(ap)

            if CODECELL_OUTPUT:
                header('RO')
                print(f'AP {LN} rolled out from FA on {today.strftime("%Y-%m-%d")}')
                print('Added to FGI object')
                line()

    shift = 0
    while shift < 3 and len(fgi.APs) > 0:
        fgi.shift = shift
        # --- STATUS UPDATES ---
        if shift == 0:
            for LN, AP in fgi.APs.items():
                AP.toB1R -= 1


            if today.weekday() < 5:
                fgi.refresh_ap_states()
                if CODECELL_OUTPUT:
                    print('Workday')
                shift = 1

            else:
                shift = 4
        
        # --- LABOR COMPLETION ---
        if shift in [1, 2]:
            fgi.set_shift(shift)
            if CODECELL_OUTPUT:
                header(f'Shift: {shift}')
                header('Labor allocation', '-')
            
            for team in FGI_TEAMS:

                worked_lns, btg_completed, btg_remaining = fgi.assign_labor(team,date=today)

                trace.record_labor(today, team, worked_lns)

                if btg_remaining > 0:
                    if CODECELL_OUTPUT:
                        print(f"Idle Time for {team}, with {btg_remaining} possible BTG completions")
            shift += 1
    
        # --- Third-shift movement (paint, compass, exit, queue execution) ---
        if shift == 3 and today not in PLANNED_BUFFER_DAYS:
            fgi.set_shift(shift)

            # --- SCHEDULED MOVES FOR PAINT ---
            fgi.schedule_paint_moves(today)

            # --- COMPASS COMPLETION / MOVE REQUESTS ---
            fgi.schedule_compass_moves(today)

            # --- SET EXIT-READY APs ---
            fgi.assign_exit_destinations()

            # --- MARK EXISTING DC EXITS BEFORE MOVEMENT ---
            fgi.mark_dc_arrivals_pending()

            # --- PROCESS MOVE QUEUE ---
            fgi.reorder_move_queue()
            move_results = fgi.execute_move_requests(date=today)

            # --- MARK DC ARRIVALS AFTER MOVEMENT ---
            fgi.mark_dc_arrivals_pending()

            if fgi.movetime_remaining <= 0:
                break

    # --- END-OF-DAY STATUS CLOSURE ---
    fgi.refresh_ap_states()
    fgi.record_day(today)
    fgi.complete_pending_exits(date=today)


    today += pd.Timedelta(days=1)


————————————————————————————————————————————————————————————————————————————————————————————————————

Location D1 added to FGI
Current locations in FGI: ['D1']
————————————————————————————————————————————————————————————————————————————————————————————————————

————————————————————————————————————————————————————————————————————————————————————————————————————

Location D2 added to FGI
Current locations in FGI: ['D1', 'D2']
————————————————————————————————————————————————————————————————————————————————————————————————————

————————————————————————————————————————————————————————————————————————————————————————————————————

Location A1 added to FGI
Current locations in FGI: ['D1', 'D2', 'A1']
————————————————————————————————————————————————————————————————————————————————————————————————————

————————————————————————————————————————————————————————————————————————————————————————————————————

Location A2 added to FGI
Current locations in FGI: ['D1', 'D2', 'A1', 'A2']
——————————————————

## 9. Post-run validation
Print delivered count, active count, pending exits, average days in system, and average days late. The active-AP status table prints only when active APs remain (expected = 0 on a clean run).

In [19]:
# Post-run validation summary.
delivery_df = fgi.get_delivery_summary_df()
active_status_df = fgi.get_active_ap_status_df()

avg_days_in_system = None
avg_days_late = None
if len(delivery_df) > 0:
    avg_days_in_system = delivery_df['Time_In_System_Days'].mean()
    avg_days_late = delivery_df['Days_Late'].mean()

if CODECELL_OUTPUT:
    print('Delivered APs:', len(delivery_df))
    print('Active APs:', len(active_status_df))
    print('Pending exits:', fgi.pendingExitLNs)
    print('Average days in system:', avg_days_in_system)
    print('Average days late:', avg_days_late)

    if len(active_status_df) > 0:
        print('Active AP status:')
        print(active_status_df)

Delivered APs: 93
Active APs: 0
Pending exits: []
Average days in system: 30.225806451612904
Average days late: -14.612903225806452


## 10. Workbook export
Builds the trace dataframes plus daily / exit / active / KPI / team-KPI / per-team BTG sheets, then writes them to `output/scheduler_trace_output.xlsx`. Required sheet list is validated after write.

In [20]:
EXPORT_PATH.mkdir(parents=True, exist_ok=True)
output_file = EXPORT_PATH / 'scheduler_trace_output.xlsx'

chickentracks_df, labor_df, moves_df, btg_dfs = trace.to_dataframes()
daily_status_df = fgi.get_daily_status_df()
delivery_df = fgi.get_delivery_summary_df()
active_status_df = fgi.get_active_ap_status_df()
kpi_summary_df = fgi.get_kpi_summary_df(trace=trace)
team_kpi_df = fgi.get_team_kpi_df(trace=trace)

required_sheets = [
    'ChickenTracks',
    'Labor Allocation',
    'Moves Per Day',
    'Daily AP Status',
    'Exit Summary',
    'Active AP Status',
    'KPI Summary',
    'Team KPIs',
    'BTG structure',
    'BTG systems',
    'BTG declam',
    'BTG test'
]

with tempfile.NamedTemporaryFile(delete=False, suffix='.xlsx') as tmp_file:
    tmp_output = Path(tmp_file.name)

try:
    with pd.ExcelWriter(tmp_output, engine='openpyxl') as writer:
        chickentracks_df.to_excel(writer, sheet_name='ChickenTracks')
        labor_df.to_excel(writer, sheet_name='Labor Allocation')
        moves_df.to_excel(writer, sheet_name='Moves Per Day')
        daily_status_df.to_excel(writer, sheet_name='Daily AP Status', index=False)
        delivery_df.to_excel(writer, sheet_name='Exit Summary', index=False)
        active_status_df.to_excel(writer, sheet_name='Active AP Status', index=False)
        kpi_summary_df.to_excel(writer, sheet_name='KPI Summary', index=False)
        team_kpi_df.to_excel(writer, sheet_name='Team KPIs', index=False)

        for team in FGI_TEAMS:
            df = btg_dfs.get(team, pd.DataFrame())
            df.to_excel(writer, sheet_name=f'BTG {team}'[:31])

    wb = load_workbook(tmp_output)

    header_fill = PatternFill('solid', fgColor='1F4E78')
    header_font = Font(color='FFFFFF', bold=True)
    index_fill = PatternFill('solid', fgColor='D9EAF7')
    thin_gray = Side(style='thin', color='BFBFBF')

    for ws in wb.worksheets:
        ws.freeze_panes = 'B2'
        ws.sheet_view.showGridLines = False

        if ws.max_row >= 1:
            for cell in ws[1]:
                cell.fill = header_fill
                cell.font = header_font
                cell.alignment = Alignment(horizontal='center', vertical='center')

        if ws.max_column >= 1:
            for cell in ws['A']:
                cell.fill = index_fill
                cell.font = Font(bold=True)
                cell.number_format = 'yyyy-mm-dd'
                cell.alignment = Alignment(horizontal='center')

        if ws.max_row >= 1 and ws.max_column >= 1:
            for row in ws.iter_rows():
                for cell in row:
                    cell.border = Border(bottom=thin_gray)
                    cell.alignment = Alignment(vertical='center', wrap_text=True)

            for col_idx, col_cells in enumerate(ws.columns, start=1):
                max_len = 0
                col_letter = get_column_letter(col_idx)

                for cell in col_cells:
                    value = cell.value
                    if value is not None:
                        max_len = max(max_len, len(str(value)))

                ws.column_dimensions[col_letter].width = min(max(max_len + 2, 12), 30)

            ws.auto_filter.ref = ws.dimensions

    wb.save(tmp_output)

    shutil.copy2(tmp_output, output_file)

    exported = pd.ExcelFile(output_file).sheet_names
    missing_sheets = [sheet for sheet in required_sheets if sheet not in exported]

    if missing_sheets:
        raise RuntimeError(f'Missing required export sheets: {missing_sheets}')

    if CODECELL_OUTPUT:
        print('Exported sheets:', exported)
        print('Delivered APs:', len(delivery_df))
        print(f'Exported trace workbook: {output_file}')

finally:
    if tmp_output.exists():
        tmp_output.unlink()


Exported sheets: ['ChickenTracks', 'Labor Allocation', 'Moves Per Day', 'Daily AP Status', 'Exit Summary', 'Active AP Status', 'KPI Summary', 'Team KPIs', 'BTG structure', 'BTG systems', 'BTG declam', 'BTG test']
Delivered APs: 93
Exported trace workbook: /Users/jishnughosh/Documents/GitHub/BSC-FGI_Scheduler/output/scheduler_trace_output.xlsx
